In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "85bd815efd86af24338a0f0c45844cb7e1e6f155"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/85bd815efd86af24338a0f0c45844cb7e1e6f155/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Self-Attention and Multiple Heads

In the examples of that section, queries came
from one sequence and keys and values from another. A decoder-only language
model has only a single sequence: to contextualize each token, queries, keys,
and values must all be derived from that sequence itself.

Consider the sentence *"Alice gave Bob the book"*. Interpreting *gave*
requires identifying the giver (*Alice*), the recipient (*Bob*), and the
object (*book*). In *self-attention*
[@Cheng.Dong.Lapata.2016; @Lin.Feng.Santos.ea.2017; @Vaswani.Shazeer.Parmar.ea.2017],
every token projects its representation into a query, a key, and a value,
allowing *gave* to query the other words in the sentence and aggregate their
values.

However, a single attention lookup produces only *one* probability
distribution per query and returns a single blended vector. How can a token
retrieve several distinct arguments without collapsing them into one average?
We first trace a single head of self-attention on our five-word sentence,
then prove an information-theoretic lower bound showing why one head cannot
recover two independent values, and finally construct *multi-head attention*
and *grouped-query attention* (GQA).

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Self-Attention

In self-attention, each token representation $\mathbf{x}_i \in \mathbb{R}^d$
(row $i$ of $\mathbf{X} \in \mathbb{R}^{n \times d}$) plays three distinct
roles via learned linear projections $\mathbf{W}_q, \mathbf{W}_k \in
\mathbb{R}^{d \times d_h}$ and $\mathbf{W}_v \in \mathbb{R}^{d \times d_v}$:
the *query* $\mathbf{x}_i\mathbf{W}_q$ specifies what the token seeks, the
*key* $\mathbf{x}_i\mathbf{W}_k$ specifies what it offers to other queries,
and the *value* $\mathbf{x}_i\mathbf{W}_v$ specifies the content returned
when selected. Applying scaled dot-product attention yields one attention
*head* of width $d_h$:

$$
\mathrm{softmax}\left(\frac{(\mathbf{X}\mathbf{W}_q)(\mathbf{X}\mathbf{W}_k)^\top}{\sqrt{d_h}}\right)\mathbf{X}\mathbf{W}_v \in \mathbb{R}^{n \times d_v}.
$$

### One Head on One Sentence

To trace the equation concretely, represent each word of
*"Alice gave Bob the book"* by a 9-dimensional vector concatenating a 4D
one-hot grammatical role (person, action, thing, or function word) and a 5D
one-hot word identity. Set $\mathbf{W}_q$ and $\mathbf{W}_k$ to read only the
4D role into $d_h = 2$ dimensions, and $\mathbf{W}_v$ to extract the 5D word
indicator:

| role | query | key |
|:--|:--|:--|
| person (Alice, Bob) | $(1, -1)$ | $(1, 1)$ |
| action (*gave*) | $(1, 1)$ | $(1, -1)$ |
| thing (book) | $(1, -1)$ | $(1, 1)$ |
| function word ("the") | $(0, 0)$ | $(0, 0)$ |

In [ ]:
tokens = ['Alice', 'gave', 'Bob', 'the', 'book']
# Role of each token: person, action, thing, function word
roles = d2l.tensor([[1.0, 0, 0, 0], [0, 1, 0, 0], [1, 0, 0, 0],
                    [0, 0, 0, 1], [0, 0, 1, 0]])
X = d2l.concat((roles, d2l.eye(5)), 1)  # (5, 9): the role, then the word
# Queries and keys read the role part (rows: person, action, thing,
# function word), values the word part
W_q = d2l.concat((d2l.tensor([[1.0, -1], [1, 1], [1, -1], [0, 0]]),
                  d2l.zeros((5, 2))))
W_k = d2l.concat((d2l.tensor([[1.0, 1], [1, -1], [1, 1], [0, 0]]),
                  d2l.zeros((5, 2))))
W_v = d2l.concat((d2l.zeros((4, 5)), d2l.eye(5)))
Q, K, V = X @ W_q, X @ W_k, X @ W_v  # (5, 2), (5, 2), (5, 5)

For *gave*, the query $(1, 1)$ has dot product $2$ with each noun's key
$(1, 1)$ and $0$ with *gave* and *"the"*. Dividing by $\sqrt{2}$ gives scaled
scores $1.41$ on each noun and $0$ elsewhere, which softmax converts into
weights $e^{1.41}/(3e^{1.41} + 2) \approx 0.29$ per noun and $0.07$ on *gave*
and *"the"* (the figure).

![One head of self-attention on "Alice gave Bob the book", with the hand-set maps of the code above ($d_h = 2$). Every token emits a query, a key, and a value. The query of *gave*, $(1, 1)$, is scored against every key by $\mathbf{q} \cdot \mathbf{k}/\sqrt{2}$; the softmax turns the scores $1.41$ and $0$ into weights $0.29$ and $0.07$; the output of *gave* is the weighted sum of the values, which mark the five words.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-self-attention.svg)

In [ ]:
attention = d2l.DotProductAttention()
Y = attention(Q[None], K[None], V[None])  # A minibatch of one sentence
weights = attention.attention_weights[0]
scores = Q[1] @ K.T / math.sqrt(Q.shape[-1])  # Scaled by sqrt(d_h)
print('scores of gave: ', np.asarray(scores).round(2))
print('weights of gave:', np.asarray(weights[1]).round(2))
print('weights of the: ', np.asarray(weights[3]).round(2))
print('output of gave: ', np.asarray(Y[0, 1]).round(2))
attention(Q[None], K[None], V[None], d2l.causal_mask(5))
print('causal weights of gave:',
      np.asarray(attention.attention_weights[0, 1]).round(2))
d2l.show_heatmaps(weights[None, None], xlabel='keys', ylabel='queries',
                  figsize=(3.2, 3.2), cmap='Blues',
                  xticklabels=tokens, yticklabels=tokens)

Conversely, each noun's query $(1, -1)$ scores $1.41$ on *gave* (weight
$0.51$) and $0$ elsewhere, while *"the"* has a zero query and attends
uniformly ($0.20$ across all five words). Applying a causal mask restricts
*gave* to *Alice* and itself, renormalizing their scores $(1.41, 0)$ to
$(0.80, 0.20)$.

### Separate Maps for Queries and Keys

Why use separate matrices $\mathbf{W}_q$ and $\mathbf{W}_k$? If a single
projection $\mathbf{u}_i = \mathbf{x}_i\mathbf{W}$ served as both query and
key, the score matrix $s_{ij} = \mathbf{u}_i^\top \mathbf{u}_j / \sqrt{d_h}$
would be symmetric and positive semidefinite, satisfying $s_{ij} \le
\sqrt{s_{ii}\, s_{jj}}$ by Cauchy--Schwarz. No token could prefer another
token over their geometric-mean self-affinity—a verb could not selectively
seek nouns without seeking itself at least as strongly. Separate query, key,
and value projections decouple *who looks*, *who is found*, and *what content
is passed on*.

## Limits of a Single Head

In our five-word sentence, *gave* blended *Alice*, *Bob*, and *book* into a
single weighted average. To quantify the bottleneck of having only one
attention distribution per query, consider the *copy-both task*: two source
positions hold independent standard Gaussian value vectors $\mathbf{v}_1,
\mathbf{v}_2 \in \mathbb{R}^{d_v}$ with fixed position-only keys, and a
single query must output their concatenation $\mathbf{t} = [\mathbf{v}_1;
\mathbf{v}_2] \in \mathbb{R}^{2d_v}$ via a linear readout.

Because the keys depend only on position, a single head assigns fixed softmax
weights $(\alpha_1, \alpha_2)$ with $\alpha_1 + \alpha_2 = 1$ and passes a
single mixture $\mathbf{m} = \alpha_1 \mathbf{v}_1 + \alpha_2 \mathbf{v}_2
\in \mathbb{R}^{d_v}$ to the linear readout.

**Proposition.** For one attention head with two fixed position-only keys,
independent standard Gaussian values $\mathbf{v}_1, \mathbf{v}_2 \in
\mathbb{R}^{d_v}$, and a linear readout, the minimum achievable relative
squared error on the copy-both task is $1/2$ for every softmax pair
$(\alpha_1, \alpha_2)$.

**Proof.** Given $\mathbf{m} = \alpha_1 \mathbf{v}_1 + \alpha_2 \mathbf{v}_2$,
the minimum mean-squared error estimator of $\mathbf{v}_1$ is the conditional
expectation $\mathbb{E}[\mathbf{v}_1 \mid \mathbf{m}] =
\tfrac{\alpha_1}{\alpha_1^2 + \alpha_2^2}\, \mathbf{m}$, which is linear in
$\mathbf{m}$ and leaves residual variance $1 - \tfrac{\alpha_1^2}{\alpha_1^2
+ \alpha_2^2} = \tfrac{\alpha_2^2}{\alpha_1^2 + \alpha_2^2}$ per coordinate.
By symmetry, estimating $\mathbf{v}_2$ leaves residual variance
$\tfrac{\alpha_1^2}{\alpha_1^2 + \alpha_2^2}$ per coordinate. Their sum is
$1$ per coordinate out of a total target variance of $2$ per coordinate,
giving a relative squared error of $1/2$ regardless of $(\alpha_1, \alpha_2)$.
$\blacksquare$

Solving the copy-both task requires two ingredients: (1) at least two
linearly independent *weightings* over the keys, and (2) sufficient *output
dimension* (room) to hold all $2d_v$ retrieved coordinates (Exercise
7). We verify the $\sqrt{1/2}
\approx 0.707$ root-mean-square relative error bound for one head, and show
that two heads with distinct weightings—even soft ones such as $(0.88, 0.12)$
and $(0.12, 0.88)$—allow a linear readout to invert the $2 \times 2$ mixing
matrix up to floating-point precision (the figure).

In [ ]:
N, d_v = 10000, 8
V1, V2 = torch.randn(N, d_v), torch.randn(N, d_v)
target = torch.cat([V1, V2], dim=1)

def readout_error(M, target):
    """Best linear readout of target from M, relative error."""
    W = torch.linalg.lstsq(M, target).solution
    return ((M @ W - target).norm() / target.norm()).item()

M = 0.7 * V1 + 0.3 * V2
print(f'one head, weights (0.70, 0.30): relative error '
      f'{readout_error(M, target):.3f}; sqrt(1/2) = {0.5 ** 0.5:.3f}')
for a in (1.0, 0.88):
    M2 = torch.cat([a * V1 + (1 - a) * V2, (1 - a) * V1 + a * V2], dim=1)
    print(f'two heads, weights ({a:.2f}, {1 - a:.2f}) and '
          f'({1 - a:.2f}, {a:.2f}): '
          f'relative error {readout_error(M2, target):.1e}')

![One weighting per query against two. A single head serves the copy-both task with one softmax weighting, so its output blends the two requested values. Two heads with different weightings supply two mixtures, from which a linear readout recovers both values.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-one-head-averages.svg)

## Multi-Head Attention

### Projected Heads

*Multi-head attention* [@Vaswani.Shazeer.Parmar.ea.2017] runs $H$
independent attention heads in parallel, each with its own query, key, and
value projections $\mathbf{W}_i^{(q)}, \mathbf{W}_i^{(k)}, \mathbf{W}_i^{(v)}
\in \mathbb{R}^{d \times d_h}$:

$$
\mathbf{Z}_i = \mathrm{softmax}\left(\frac{(\mathbf{X}\mathbf{W}_i^{(q)})(\mathbf{X}\mathbf{W}_i^{(k)})^\top}{\sqrt{d_h}}\right)\mathbf{X}\mathbf{W}_i^{(v)} \in \mathbb{R}^{n \times d_h}, \qquad i = 1, \ldots, H,
$$

and concatenates their outputs before applying a final linear projection
$\mathbf{W}_o \in \mathbb{R}^{H d_h \times d}$ (the figure):

$$
[\mathbf{Z}_1, \ldots, \mathbf{Z}_H]\, \mathbf{W}_o \in \mathbb{R}^{n \times d}.
$$

![Multi-head attention. Every head reads the same queries, keys, and values, projects them into its own subspace, and attends there; the head outputs are concatenated and recombined by one output projection.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-multi-head.svg)

### Parameter and FLOP Counts

Setting the head width to $d_h = d/H$ keeps the total dimension across all
heads equal to $H d_h = d$. Stacked side by side, the $H$ query projections
form a single $d \times d$ matrix, as do the key, value, and output
projections. Counting one multiply--add as $2$ FLOPs, a multi-head
self-attention layer over $n$ tokens requires

$$
\underbrace{8nd^2}_{\textrm{projections}} \;+\; \underbrace{4n^2 d}_{\textrm{scores and mixing}} \quad \textrm{FLOPs,} \qquad 4d^2 \ \textrm{parameters.}
$$

For one layer of GPT-2 small ($d = 768$, $n = 1{,}024$), this amounts to
$4.8$ GFLOPs for projections, $3.2$ GFLOPs for scores and value mixing, and
$2.4 \times 10^6$ parameters. Remarkably, neither the parameter count nor the
leading FLOP count depends on the number of heads $H$: splitting $d$ into
$H$ heads shrinks each head's inner product by $1/H$ while running $H$ of
them (only the $O(Hn^2)$ softmax overhead scales with $H$). Conversely, $d_h$
cannot be made arbitrarily small: at $d_h = 1$, scalar scores $q k_j$ are
monotone in $k_j$, so a query can focus only on the extremal keys (Exercise
2). In practice, $d_h \in \{64, 128\}$ is
standard.

### Comparison with Convolution and Recurrence

the figure and the table compare
1D convolution (that section), recurrence (that section), and
self-attention as sequence-to-sequence layers over $n$ tokens of width $d$.

![Sequence-layer connectivity for five tokens. Highlighted edges show how the inputs reach the third output in one layer: a convolution with kernel size $k = 3$ sees its neighbors, a recurrence passes information step by step along the chain, and self-attention connects every position directly. Below each panel are the path length between distant tokens and whether the outputs are computed in parallel.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-cnn-rnn-self-attention.svg)

<!-- tbl-caption:Per-layer cost of mapping $n$ tokens of dimension $d$ to $n$ tokens of dimension $d$. -->

| layer type | complexity | sequential operations | maximum path length |
|:--|:--|:--|:--|
| convolution (kernel $k$) | $O(knd^2)$ | $O(1)$ | $O(n/k)$ |
| recurrence | $O(nd^2)$ | $O(n)$ | $O(n)$ |
| self-attention | $O(nd^2+n^2d)$ | $O(1)$ | $O(1)$ |

Self-attention combines $O(1)$ sequential steps (full training parallelism)
with $O(1)$ maximum path length between any pair of tokens, eliminating the
$O(n)$-step recurrent gradient chain (that section). The price is the
quadratic $4n^2 d$ score-and-mix term, which exceeds the $8nd^2$ projection
cost once $n > 2d$ ($n > 1{,}536$ for GPT-2 small).

### Implementation

In `MultiHeadAttention`, a single linear layer computes all $H$ head
projections at once, and `split_heads` reshapes `(batch, length, H * d_h)`
into `(batch, H, length, d_h)` so that `DotProductAttention` executes all
heads in one batched operation (the figure). Projections omit
additive biases by default (`bias=False`), following modern transformer
conventions. The parameter `num_kv_heads` supports key--value head sharing
(that section).

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention with heads as an explicit axis; num_kv_heads <
    num_heads shares each key-value head across a group of query heads."""
    def __init__(self, num_hiddens, num_heads, dropout=0.0, bias=False,
                 num_kv_heads=None):
        super().__init__()
        assert num_hiddens % num_heads == 0, 'heads must divide num_hiddens'
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads or num_heads
        assert num_heads % self.num_kv_heads == 0
        kv_hiddens = self.num_kv_heads * (num_hiddens // num_heads)
        self.attention = d2l.DotProductAttention(dropout)
        self.W_q = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        self.W_k = nn.Linear(num_hiddens, kv_hiddens, bias=bias)
        self.W_v = nn.Linear(num_hiddens, kv_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, mask=None):
        # queries: (batch, no. of queries, num_hiddens); keys and values:
        # (batch, no. of key-value pairs, num_hiddens). mask: boolean,
        # broadcastable to (batch, no. of queries, no. of keys), shared by
        # all heads, or to (batch, num_heads, no. of queries, no. of keys)
        q = self.split_heads(self.W_q(queries), self.num_heads)
        k = self.split_heads(self.W_k(keys), self.num_kv_heads)
        v = self.split_heads(self.W_v(values), self.num_kv_heads)
        group = self.num_heads // self.num_kv_heads
        k, v = k.repeat_interleave(group, 1), v.repeat_interleave(group, 1)
        if mask is not None and mask.dim() == 3:
            mask = mask[:, None]           # The same mask for every head
        # Shape of Y: (batch, num_heads, no. of queries, head width)
        Y = self.attention(q, k, v, mask)
        return self.W_o(self.merge_heads(Y))

    @property
    def attention_weights(self):
        """Weights of the last call: (batch, num_heads, queries, keys)."""
        return self.attention.attention_weights

![Heads as slices of one projection. The projected queries of $n$ tokens are cut into $H$ column slices of width $d_h = d/H$, which become a separate head axis; each head attends on its own, and the head outputs are placed side by side again before the output projection $\mathbf{W}_o$. Keys and values are split in the same way.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-head-split.svg)

In [ ]:
@d2l.add_to_class(MultiHeadAttention)
def split_heads(self, X, num_heads):
    """(batch, length, num_heads * width) -> (batch, num_heads, length,
    width)."""
    return X.reshape(X.shape[0], X.shape[1], num_heads, -1).transpose(1, 2)

@d2l.add_to_class(MultiHeadAttention)
def merge_heads(self, X):
    """Reverse split_heads: (batch, num_heads, length, width) -> (batch,
    length, num_heads * width)."""
    return X.transpose(1, 2).reshape(X.shape[0], X.shape[2], -1)

### Learned Heads on the Copy-Both Task

We now test `MultiHeadAttention` of width $d = 16$ on the copy-both task
($d_v = 8$, padded with zeros to width $16$; keys $\mathbf{e}_1,
\mathbf{e}_2$ and query $\mathbf{e}_3$).

In [ ]:
def copy_both_data(n, d=16, d_v=8):
    """One query, two position-only keys, values of width d_v padded to d."""
    V = torch.randn(n, 2, d_v)
    keys = torch.eye(d)[:2].expand(n, 2, d)    # e_1, e_2 mark the positions
    query = torch.eye(d)[2].expand(n, 1, d)    # e_3 asks for both values
    values = F.pad(V, (0, d - d_v))
    return query, keys, values, V.reshape(n, 1, 2 * d_v)

def train_copy_both(num_heads, num_kv_heads=None, num_steps=2000):
    model = MultiHeadAttention(16, num_heads, num_kv_heads=num_kv_heads)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(num_steps):
        query, keys, values, target = copy_both_data(256)
        loss = ((model(query, keys, values) - target) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

def evaluate_copy_both(model, n=10000):
    """Relative error on fresh examples; each head's weight on v_1."""
    query, keys, values, target = copy_both_data(n)
    with torch.inference_mode():
        err = (model(query, keys, values) - target).norm() / target.norm()
    return err.item(), model.attention_weights[0, :, 0, 0].tolist()

In [ ]:
print('H  run  parameters  error    weight on v_1, per head')
for num_heads in (1, 2, 4):
    for run in (1, 2):
        model = train_copy_both(num_heads)
        err, alpha_1 = evaluate_copy_both(model)
        print(f'{num_heads}  {run:3d}  {d2l.num_params(model):10d}  '
              f'{err:<#8.2g} ' + ' '.join(f'{a:.2f}' for a in alpha_1))

All configurations have $4 \times 16^2 = 1{,}024$ parameters. As predicted by
our proposition, $H = 1$ stalls at the $\sqrt{1/2} \approx 0.71$ bound,
whereas $H \in \{2, 4\}$ learns distinct head weightings and reduces the
error to $\sim 10^{-7}$.

### Shared Keys and Values

During autoregressive generation, past keys and values are cached in memory
(that section). *Grouped-query attention* (GQA)
[@Ainslie.Lee-Thorp.Jong.ea.2023] reduces this cache by a factor of
$H / H_\textrm{kv}$ by sharing each key--value head across a group of $H /
H_\textrm{kv}$ query heads; setting $H_\textrm{kv} = 1$ gives *multi-query
attention* (MQA) [@Shazeer.2019].

On the copy-both task, the shared value heads span at most $H_\textrm{kv}
d_h$ coordinates per value, so recovering a $d_v$-dimensional value requires
$H_\textrm{kv} d_h \ge d_v$ (Exercise
8). With $d = 16$ and $d_v = 8$,
layouts $(H, H_\textrm{kv}) = (2, 1)$ and $(4, 2)$ have $H_\textrm{kv} d_h =
8$ and solve the task to fp32 precision, whereas $(4, 1)$ has $H_\textrm{kv}
d_h = 4$ and returns to the $0.71$ bottleneck despite learning distinct
weightings across its four query heads:

In [ ]:
print('H  H_kv  run  parameters  error    weight on v_1, per head')
for num_heads, num_kv_heads in ((2, 1), (4, 2), (4, 1)):
    for run in (1, 2):
        model = train_copy_both(num_heads, num_kv_heads)
        err, alpha_1 = evaluate_copy_both(model)
        print(f'{num_heads}  {num_kv_heads:4d}  {run:3d}  '
              f'{d2l.num_params(model):10d}  {err:<#8.2g} '
              + ' '.join(f'{a:.2f}' for a in alpha_1))

## Cross-Attention

While self-attention passes the same sequence $\mathbf{X}$ as queries, keys,
and values (`attention(X, X, X)`), *cross-attention* draws queries from a
target sequence $\mathbf{A}$ and keys and values from a source sequence
$\mathbf{B}$ (`attention(A, B, B)`), producing one output row per token of
$\mathbf{A}$. Encoder--decoder transformers use cross-attention to read an
encoded source sentence during translation, and multimodal models use it to
let text tokens query visual features [@alayrac2022flamingo].

Even with multiple heads, however, self-attention remains blind to word
order: swapping *Alice* and *Bob* in *"Alice gave Bob the book"* leaves the
output at *gave* unchanged. Injecting sequence order is the subject of
that section.

## Summary

* Self-attention projects each token of a sequence into a query, a key, and a
  value, connecting all pairs of positions in $O(1)$ sequential steps and
  $O(1)$ path length at $O(nd^2 + n^2d)$ FLOPs.
* A single head forms only one convex mixture of values per query; on the
  copy-both task, any single head incurs a relative squared error of at least
  $1/2$.
* Multi-head attention runs $H$ heads of width $d_h = d/H$ in parallel and
  combines them via $\mathbf{W}_o$, requiring $4d^2$ parameters and
  $8nd^2 + 4n^2d$ FLOPs independent of $H$.
* Grouped-query and multi-query attention share $H_\textrm{kv} < H$
  key--value heads across query groups to shrink generation caches by
  $H/H_\textrm{kv}$, preserving capacity so long as $H_\textrm{kv} d_h$ has
  room for the retrieved values.

## Exercises

1. **Self-attention by hand.** Three tokens have the inputs
   $\mathbf{x}_1 = (1, 0)$, $\mathbf{x}_2 = (0, 1)$, and $\mathbf{x}_3 =
   (1, 1)$. One head of width $d_h = 2$ uses
   $\mathbf{W}_q = \sqrt{2}\,\mathbf{I}$, the swap
   $\mathbf{W}_k = \left(\begin{smallmatrix} 0 & 1 \\ 1 & 0
   \end{smallmatrix}\right)$, and $\mathbf{W}_v = \mathbf{I}$, so that the
   factor $\sqrt{2}$ cancels the scale $1/\sqrt{d_h}$.
    1. Compute the queries, keys, and values, the $3 \times 3$ matrix of
       scaled scores, the weights, and the three outputs, each to two
       decimals. Which token does the first token attend to least, and why?
    1. Apply a causal mask and recompute. Which rows change, and why does
       the last row stay as it was?
    1. Replace $\mathbf{W}_k$ by the identity, so that each key points the
       same way as the query of its token. Recompute the weights of the
       first token. Why can no key of the same length as $\mathbf{x}_1$
       score higher than the first token's own key, and why does the key of
       the third token score as high anyway?
    1. Check your numbers with `d2l.DotProductAttention`.

1. [code] **Narrow heads.** With $d_h = d/H$, the leading count of
   the equation does not depend on $H$, so the number of heads
   looks free up to $H = d$. Consider one head with scores
   $\mathbf{q}^\top \mathbf{k}_j / \sqrt{d_h}$ over keys $\mathbf{k}_1,
   \ldots, \mathbf{k}_m \in \mathbb{R}^{d_h}$ and no mask. Call a query sharp
   on key $j$ when its weights concentrate on $\mathbf{k}_j$ as the query
   is scaled by a factor that grows without bound.
    1. For $d_h = 1$ and distinct scalar keys, show that every query's weights
       are monotone in $k_j$. Conclude that the queries of such a head can
       be sharp only on the largest or the smallest key.
    1. Show that for $d_h = 2$ the keys and queries can be chosen so that
       each of $n$ queries is sharp on a key of its own choosing, for every
       assignment of queries to keys.
    1. Use these two results to state which attention patterns a layer
       gives up as $H$ approaches $d$.
    1. Time the forward pass of `MultiHeadAttention` with dropout 0 for
       self-attention with $d = 256$ and $n = 512$ on a minibatch of $8$
       sequences for $H \in \{1, 2, 4, 8, 16, 32, 64\}$. Make a warm-up
       call first and wait for the result before stopping the clock.
       Report the ratio of the slowest to the fastest configuration.
       Identify the cost that grows with $H$ and that
       the equation leaves out, and check that it accounts
       for the trend.

1. [code] **Heads as slices.** `MultiHeadAttention` computes one projection
   of width $d$ for each of the queries, keys, and values and splits it
   into heads with `split_heads`.
    1. Head $i$ owns a slice of the output features of `W_q`, `W_k`, and
       `W_v`. Determine which input features of `W_o` receive that head's
       output. Write a forward pass that loops over the heads, applies the
       sliced projections, and calls `d2l.DotProductAttention` once per
       head. Verify on random inputs with a padding mask that it matches
       `MultiHeadAttention` with dropout 0, and extend your loop to
       `num_kv_heads < num_heads`.
    1. Replace the body of `split_heads` by
       `X.reshape(X.shape[0], num_heads, X.shape[1], -1)`, which returns a
       tensor of the same shape. Describe which numbers each "head" now
       holds, show on random inputs that the output changes, and explain why
       no shape check can detect the mistake. Which input lengths and widths,
       if any, make the two versions agree?

1. **Permuting the source.** In cross-attention, the queries come from
   $\mathbf{A} \in \mathbb{R}^{n \times d}$ and the keys and values from
   $\mathbf{B} \in \mathbb{R}^{m \times d}$, neither carrying positional
   information. Let $\boldsymbol{\Pi}$ be an $m \times m$ permutation
   matrix.
    1. Show that replacing $\mathbf{B}$ by $\boldsymbol{\Pi}\mathbf{B}$ in
       both the key and the value argument leaves the output of
       `MultiHeadAttention` unchanged when no mask is applied, and that
       replacing it in the key argument alone changes the output in general.
    1. Now let a padding mask mark only the first $\ell$ rows of
       $\mathbf{B}$ as valid. Which permutations leave the output unchanged
       when the mask stays fixed?

1. [code] **Nonlinear readouts.** ● For Gaussian values, the proof of the
   proposition in that section shows that the
   conditional mean $\mathbb{E}[\mathbf{v}_1 \mid \mathbf{m}]$ is linear in
   the mixture. Hence no readout, linear or not, improves on the relative
   error $\sqrt{1/2}$ that `readout_error` measures.
    1. Train a small multilayer perceptron to predict `target` from the
       mixture $\mathbf{m} = a\,\mathbf{v}_1 + (1 - a)\,\mathbf{v}_2$
       formed from `V1` and `V2`, for $a = 0.5$ and $a = 0.7$. Compare its
       relative error on held-out samples with that of the linear readout.
    1. Now draw the values uniformly from $\{-1, +1\}^{d_v}$. For $a = 0.5$,
       compute $\mathbb{E}[\mathbf{v}_1 \mid \mathbf{m}]$ coordinate by
       coordinate, show that it is again linear in $\mathbf{m}$, and
       conclude that the error stays at $\sqrt{1/2}$ for every readout.
    1. For $a = 0.7$ and the same discrete values, show that the map
       $(v_{1j}, v_{2j}) \mapsto m_j$ is injective. Train the perceptron
       again and report its held-out error. Explain why this escape from
       the bound does not contradict the proposition.

1. [code] **Nearly identical heads.** ● Two heads with weights
   $(0.5 + \epsilon, 0.5 - \epsilon)$ and $(0.5 - \epsilon, 0.5 +
   \epsilon)$ supply two distinct mixtures, so the linear readout inverts
   them exactly for every $\epsilon > 0$.
    1. Write the $2 \times 2$ mixing matrix, compute its eigenvalues and
       eigenvectors, and express its condition number as a function of
       $\epsilon$.
    1. Add independent Gaussian noise of standard deviation $\sigma = 0.01$
       to both mixtures and measure the error with `readout_error` for
       $\epsilon \in \{0.4, 0.1, 0.03, 0.01, 0.003, 0.001\}$. Plot the
       error against $\epsilon$ on logarithmic axes, read off the slope
       where the curve is straight, and explain it with the condition
       number from (a).
    1. Every head of a freshly initialized layer weights the two positions
       of the copy-both task close to $(0.5, 0.5)$. Modify
       `train_copy_both` to record the relative error and the weights of
       each head every 20 steps, and plot both for $H = 2$. Use the
       conditioning of the mixing matrix to explain the shape of the error
       curve in the first few hundred steps. Repeat for $H = 8$ and sort the
       final heads into those that are nearly one-hot, those that are soft
       but distinct, and those that nearly duplicate another head.

1. **What the bound counts.** ● The proposition of
   that section fixes the softmax weights, but
   its error bound rests on a weaker fact: the readout receives a single
   vector $\mathbf{m} \in \mathbb{R}^{d_v}$ and must produce the target
   $\mathbf{t} \in \mathbb{R}^{2d_v}$.
    1. Let $\mathbf{m} \in \mathbb{R}^{d_v}$ be any function of
       $\mathbf{v}_1$ and $\mathbf{v}_2$, including a mixture whose weights
       depend on the values, and let $\mathbf{W}_o \in \mathbb{R}^{d_v
       \times 2d_v}$ be any matrix. Show that
       $$\mathbb{E}\,\|\mathbf{m}\mathbf{W}_o - \mathbf{t}\|^2 \geq \tfrac{1}{2}\, \mathbb{E}\,\|\mathbf{t}\|^2.$$
       Conclude that content-dependent keys, more layers, or a residual
       connection leave the bound in place as long as the readout is linear
       and receives $d_v$ numbers.
    1. A layer of width $d$ with two heads of width $d_h = d/2$ receives the
       values $\mathbf{v}_1, \mathbf{v}_2 \in \mathbb{R}^{d/2}$ padded with
       zeros to width $d$. Give the attention weights of each head, value
       projections $\mathbf{W}_i^{(v)}$, and an output projection
       $\mathbf{W}_o$ that solve the task exactly. Compare your weights with
       those printed in that section.
    1. Generalize the proposition to $k$ source positions with independent
       standard Gaussian values $\mathbf{v}_1, \ldots, \mathbf{v}_k \in
       \mathbb{R}^{d_v}$ and the target $[\mathbf{v}_1; \ldots;
       \mathbf{v}_k]$. Each of $H$ heads applies fixed weights to the
       values, the weights of head $i$ form row $i$ of a matrix
       $\mathbf{A} \in \mathbb{R}^{H \times k}$, and a linear readout sees
       all $H$ mixtures. Show that the best relative squared error is
       $$1 - \frac{\operatorname{rank}(\mathbf{A})}{k},$$
       and that softmax weights, whose entries are all positive, can attain
       $\operatorname{rank}(\mathbf{A}) = \min(H, k)$.

1. [code] **Sharing keys and values on the copy-both task.** ● Let a layer
   of width $d$ have $H$ query heads of width $d_h = d/H$ and
   $H_\textrm{kv} < H$ key--value heads.
    1. Use the counting of Exercise
       7 to show that it can solve
       the task only if $H_\textrm{kv} d_h \ge d_v$, and that this condition
       suffices.
    1. Train a layer with `train_copy_both(2, 1)`, in which two query heads
       share one key--value head. Find the parameters that differ between
       the two heads and those they share, and explain from the copy-both
       task why the difference must lie where you found it.

[Discussions](https://d2l.discourse.group/t/36770)